<a href="https://colab.research.google.com/github/jarekwan/PROJEKT_SCANNER/blob/main/6filtry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/projekt_test', exist_ok=True)

print("folder ready")

In [ ]:
%%writefile /content/drive/MyDrive/projekt_test/modul_filtry.py

from __future__ import annotations

import json
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime
from enum import StrEnum
from pathlib import Path
from typing import Any, Callable, Final, Protocol, Self


FOLDER_PROJEKTU: Final[Path] = Path(
    "/content/drive/MyDrive/projekt_test"
)


class TypFiltra(StrEnum):
    SMA = "sma"
    MOMENTUM = "momentum"
    ADVANCED_MOMENTUM = "advanced_momentum"
    ZMIENNOSC = "zmiennosc"


class StatusFiltra(StrEnum):
    PASSED = "passed"
    FAILED = "failed"


@dataclass(
    frozen=True,
    slots=True,
    kw_only=True
)
class SpolkaDoFiltrow:
    ticker: str

    srednia_kroczaca: float
    stopa_zwrotu: float
    sredni_wolumen: float
    zmiennosc: float

    liczba_notowan: int

    cena_minimum: float
    cena_maksimum: float
    cena_srednia: float


@dataclass(
    frozen=True,
    slots=True,
    kw_only=True
)
class WynikFiltra:
    typ_filtra: TypFiltra

    status: StatusFiltra

    wartosc: float

    prog: float

    opis: str

    timestamp: datetime = field(
        default_factory=datetime.now,
        compare=False,
        repr=False
    )


@dataclass(
    slots=True,
    kw_only=True
)
class WynikLancuchaFiltrow:
    ticker: str

    wyniki: list[WynikFiltra] = field(
        default_factory=list
    )

    wszystkie_spelnione: bool = field(
        init=False
    )

    liczba_spelnionych: int = field(
        init=False
    )

    liczba_niespelnionych: int = field(
        init=False
    )

    def __post_init__(self) -> None:

        self.liczba_spelnionych = sum(
            1
            for wynik in self.wyniki
            if wynik.status == StatusFiltra.PASSED
        )

        self.liczba_niespelnionych = sum(
            1
            for wynik in self.wyniki
            if wynik.status == StatusFiltra.FAILED
        )

        self.wszystkie_spelnione = (
            self.liczba_niespelnionych == 0
        )


class FilterProtocol(Protocol):

    def ocen(
        self,
        spolka: SpolkaDoFiltrow
    ) -> WynikFiltra:
        ...

    def __call__(
        self,
        spolka: SpolkaDoFiltrow
    ) -> WynikFiltra:
        ...


FiltrCallable = Callable[
    [SpolkaDoFiltrow],
    WynikFiltra
]


class BaseFilter(ABC):

    def __init__(self) -> None:

        self._next_filter: BaseFilter | None = None

    def set_next(
        self,
        filtr: BaseFilter
    ) -> Self:

        self._next_filter = filtr

        return self

    @abstractmethod
    def ocen(
        self,
        spolka: SpolkaDoFiltrow
    ) -> WynikFiltra:

        raise NotImplementedError

    def __call__(
        self,
        spolka: SpolkaDoFiltrow
    ) -> WynikFiltra:

        return self.ocen(
            spolka
        )

    def handle(
        self,
        spolka: SpolkaDoFiltrow,
        wyniki: list[WynikFiltra]
    ) -> list[WynikFiltra]:

        wynik: WynikFiltra = self.ocen(
            spolka
        )

        wyniki.append(
            wynik
        )

        if (
            wynik.status == StatusFiltra.PASSED
            and self._next_filter is not None
        ):
            return self._next_filter.handle(
                spolka,
                wyniki
            )

        return wyniki


class OpisFiltraMixin:

    def opis_konfiguracji(
        self
    ) -> str:

        dane: list[str] = []

        for nazwa, wartosc in vars(
            self
        ).items():

            if not nazwa.startswith("_"):
                dane.append(
                    f"{nazwa}={wartosc}"
                )

        return ", ".join(
            dane
        )


class FiltrSMA(
    OpisFiltraMixin,
    BaseFilter
):

    def __init__(
        self,
        minimalna_relacja: float = 1.0
    ) -> None:

        super().__init__()

        self.minimalna_relacja = (
            minimalna_relacja
        )

    def ocen(
        self,
        spolka: SpolkaDoFiltrow
    ) -> WynikFiltra:

        if spolka.cena_srednia == 0:

            relacja: float = 0.0

        else:

            relacja = (
                spolka.srednia_kroczaca
                / spolka.cena_srednia
            )

        passed: bool = (
            relacja
            >= self.minimalna_relacja
        )

        return WynikFiltra(
            typ_filtra=TypFiltra.SMA,

            status=(
                StatusFiltra.PASSED
                if passed
                else StatusFiltra.FAILED
            ),

            wartosc=relacja,

            prog=self.minimalna_relacja,

            opis=(
                "srednia kroczaca / "
                "srednia cena historyczna"
            )
        )


class FiltrMomentum(
    OpisFiltraMixin,
    BaseFilter
):

    def __init__(
        self,
        minimalne_momentum: float = 0.0
    ) -> None:

        super().__init__()

        self.minimalne_momentum = (
            minimalne_momentum
        )

    def ocen(
        self,
        spolka: SpolkaDoFiltrow
    ) -> WynikFiltra:

        wartosc: float = (
            spolka.stopa_zwrotu
        )

        passed: bool = (
            wartosc
            >= self.minimalne_momentum
        )

        return WynikFiltra(
            typ_filtra=TypFiltra.MOMENTUM,

            status=(
                StatusFiltra.PASSED
                if passed
                else StatusFiltra.FAILED
            ),

            wartosc=wartosc,

            prog=self.minimalne_momentum,

            opis=(
                "prosta stopa zwrotu "
                "jako podstawowe momentum"
            )
        )


class AdvancedMomentumFilter(
    FiltrMomentum
):

    def __init__(
        self,
        minimalne_momentum: float = 2.0,
        minimalny_wolumen: float = 100_000.0
    ) -> None:

        super().__init__(
            minimalne_momentum=(
                minimalne_momentum
            )
        )

        self.minimalny_wolumen = (
            minimalny_wolumen
        )

    def ocen(
        self,
        spolka: SpolkaDoFiltrow
    ) -> WynikFiltra:

        podstawowy: WynikFiltra = (
            super().ocen(
                spolka
            )
        )

        momentum_ok: bool = (
            podstawowy.status
            == StatusFiltra.PASSED
        )

        wolumen_ok: bool = (
            spolka.sredni_wolumen
            >= self.minimalny_wolumen
        )

        passed: bool = (
            momentum_ok
            and wolumen_ok
        )

        return WynikFiltra(
            typ_filtra=(
                TypFiltra.ADVANCED_MOMENTUM
            ),

            status=(
                StatusFiltra.PASSED
                if passed
                else StatusFiltra.FAILED
            ),

            wartosc=(
                spolka.stopa_zwrotu
            ),

            prog=(
                self.minimalne_momentum
            ),

            opis=(
                "momentum + minimalny "
                f"wolumen {self.minimalny_wolumen}"
            )
        )


class FiltrZmiennosci(
    OpisFiltraMixin,
    BaseFilter
):

    def __init__(
        self,
        maksymalna_zmiennosc: float = 5.0
    ) -> None:

        super().__init__()

        self.maksymalna_zmiennosc = (
            maksymalna_zmiennosc
        )

    def ocen(
        self,
        spolka: SpolkaDoFiltrow
    ) -> WynikFiltra:

        wartosc: float = (
            spolka.zmiennosc
        )

        passed: bool = (
            wartosc
            <= self.maksymalna_zmiennosc
        )

        return WynikFiltra(
            typ_filtra=(
                TypFiltra.ZMIENNOSC
            ),

            status=(
                StatusFiltra.PASSED
                if passed
                else StatusFiltra.FAILED
            ),

            wartosc=wartosc,

            prog=self.maksymalna_zmiennosc,

            opis=(
                "zmiennosc nie moze "
                "przekroczyc limitu"
            )
        )


KONFIGURACJA_FILTROW: Final[
    list[dict[str, Any]]
] = [
    {
        "typ": TypFiltra.SMA,
        "parametry": {
            "minimalna_relacja": 1.0
        }
    },

    {
        "typ": TypFiltra.MOMENTUM,
        "parametry": {
            "minimalne_momentum": 0.0
        }
    },

    {
        "typ":
            TypFiltra.ADVANCED_MOMENTUM,

        "parametry": {
            "minimalne_momentum": 2.0,
            "minimalny_wolumen": 100_000.0
        }
    },

    {
        "typ": TypFiltra.ZMIENNOSC,
        "parametry": {
            "maksymalna_zmiennosc": 5.0
        }
    }
]


FABRYKA_FILTROW: Final[
    dict[TypFiltra, Callable[..., BaseFilter]]
] = {

    TypFiltra.SMA:
        FiltrSMA,

    TypFiltra.MOMENTUM:
        FiltrMomentum,

    TypFiltra.ADVANCED_MOMENTUM:
        AdvancedMomentumFilter,

    TypFiltra.ZMIENNOSC:
        FiltrZmiennosci
}


def zbuduj_filtry(
    konfiguracja: list[dict[str, Any]]
) -> list[BaseFilter]:

    filtry: list[BaseFilter] = []

    for pozycja in konfiguracja:

        typ: TypFiltra = (
            pozycja["typ"]
        )

        parametry: dict[str, Any] = (
            pozycja.get(
                "parametry",
                {}
            )
        )

        klasa_filtra: Callable[
            ...,
            BaseFilter
        ] = FABRYKA_FILTROW[typ]

        filtr: BaseFilter = (
            klasa_filtra(
                **parametry
            )
        )

        filtry.append(
            filtr
        )

    return filtry


def zbuduj_lancuch(
    filtry: list[BaseFilter]
) -> BaseFilter:

    if not filtry:
        raise ValueError(
            "lista filtrow nie moze byc pusta"
        )

    for aktualny, nastepny in zip(
        filtry,
        filtry[1:]
    ):

        aktualny.set_next(
            nastepny
        )

    return filtry[0]


def wczytaj_dane_z_modulu_5(
    ticker: str,
    folder: Path = FOLDER_PROJEKTU
) -> dict[str, Any]:

    ticker = ticker.strip().upper()

    plik: Path = (
        folder
        / f"{ticker}_obliczenia.json"
    )

    if not plik.exists():

        raise FileNotFoundError(
            f"brak pliku z modulu 5: "
            f"{plik}"
        )

    with open(
        plik,
        "r",
        encoding="utf-8"
    ) as f:

        dane: Any = json.load(
            f
        )

    if not isinstance(
        dane,
        dict
    ):
        raise ValueError(
            "dane modulu 5 "
            "musza byc slownikiem"
        )

    if "ticker" not in dane:
        raise ValueError(
            "brak ticker"
        )

    if "obliczenia" not in dane:
        raise ValueError(
            "brak sekcji obliczenia"
        )

    return dane


def dane_na_spolke(
    dane: dict[str, Any]
) -> SpolkaDoFiltrow:

    obliczenia: dict[str, Any] = (
        dane["obliczenia"]
    )

    zakres: dict[str, Any] = (
        obliczenia["zakres_cen"]
    )

    return SpolkaDoFiltrow(
        ticker=str(
            dane["ticker"]
        ).strip().upper(),

        srednia_kroczaca=float(
            obliczenia[
                "srednia_kroczaca_20"
            ]
        ),

        stopa_zwrotu=float(
            obliczenia[
                "stopa_zwrotu_proc"
            ]
        ),

        sredni_wolumen=float(
            obliczenia[
                "sredni_wolumen_20"
            ]
        ),

        zmiennosc=float(
            obliczenia[
                "zmiennosc_proc"
            ]
        ),

        liczba_notowan=int(
            obliczenia[
                "liczba_notowan"
            ]
        ),

        cena_minimum=float(
            zakres["minimum"]
        ),

        cena_maksimum=float(
            zakres["maksimum"]
        ),

        cena_srednia=float(
            zakres["srednia"]
        )
    )


def wykonaj_pojedyncza_strategie(
    spolka: SpolkaDoFiltrow,
    strategia: FiltrCallable
) -> WynikFiltra:

    return strategia(
        spolka
    )


def wykonaj_lancuch(
    spolka: SpolkaDoFiltrow,
    pierwszy_filtr: BaseFilter
) -> WynikLancuchaFiltrow:

    wyniki: list[
        WynikFiltra
    ] = []

    pierwszy_filtr.handle(
        spolka,
        wyniki
    )

    return WynikLancuchaFiltrow(
        ticker=spolka.ticker,
        wyniki=wyniki
    )


def wynik_do_dict(
    wynik: WynikLancuchaFiltrow,
    dane_modulu_5: dict[str, Any]
) -> dict[str, Any]:

    return {

        "ticker":
            wynik.ticker,

        "dane_wejsciowe":
            dane_modulu_5,

        "podsumowanie_filtrow": {

            "wszystkie_spelnione":
                wynik.wszystkie_spelnione,

            "liczba_spelnionych":
                wynik.liczba_spelnionych,

            "liczba_niespelnionych":
                wynik.liczba_niespelnionych
        },

        "filtry": [

            {
                "typ":
                    filtr.typ_filtra.value,

                "status":
                    filtr.status.value,

                "wartosc":
                    filtr.wartosc,

                "prog":
                    filtr.prog,

                "opis":
                    filtr.opis,

                "timestamp":
                    filtr.timestamp.isoformat()
            }

            for filtr in wynik.wyniki
        ]
    }


def zapisz_wynik(
    ticker: str,
    dane: dict[str, Any],
    folder: Path = FOLDER_PROJEKTU
) -> Path:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

    ticker = ticker.strip().upper()

    plik: Path = (
        folder
        / f"{ticker}_filtry.json"
    )

    with open(
        plik,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            dane,
            f,
            ensure_ascii=False,
            indent=2
        )

    return plik


def run() -> None:

    ticker: str = input(
        "podaj ticker: "
    ).strip().upper()

    print(
        "\nwczytywanie danych "
        "zapisanych przez modul 5..."
    )

    dane: dict[str, Any] = (
        wczytaj_dane_z_modulu_5(
            ticker
        )
    )

    spolka: SpolkaDoFiltrow = (
        dane_na_spolke(
            dane
        )
    )

    print(
        "utworzono obiekt do filtrowania:",
        spolka.ticker
    )

    filtry: list[BaseFilter] = (
        zbuduj_filtry(
            KONFIGURACJA_FILTROW
        )
    )

    print(
        "\nUTWORZONE FILTRY:"
    )

    for filtr in filtry:

        print(
            "-",
            type(filtr).__name__
        )

    advanced: bool = any(
        isinstance(
            filtr,
            AdvancedMomentumFilter
        )
        for filtr in filtry
    )

    print(
        "\nCzy lancuch zawiera "
        "AdvancedMomentumFilter:",
        advanced
    )

    pierwszy_filtr: BaseFilter = (
        zbuduj_lancuch(
            filtry
        )
    )

    wynik: WynikLancuchaFiltrow = (
        wykonaj_lancuch(
            spolka,
            pierwszy_filtr
        )
    )

    print(
        "\nWYNIKI FILTROW:"
    )

    for rezultat in wynik.wyniki:

        print(
            rezultat.typ_filtra.value,
            "->",
            rezultat.status.value,
            "| wartosc:",
            round(
                rezultat.wartosc,
                4
            ),
            "| prog:",
            rezultat.prog
        )

    print(
        "\nLICZBA SPELNIONYCH:",
        wynik.liczba_spelnionych
    )

    print(
        "LICZBA NIESPELNIONYCH:",
        wynik.liczba_niespelnionych
    )

    print(
        "WSZYSTKIE SPELNIONE:",
        wynik.wszystkie_spelnione
    )

    wynik_json: dict[str, Any] = (
        wynik_do_dict(
            wynik,
            dane
        )
    )

    plik: Path = zapisz_wynik(
        ticker,
        wynik_json
    )

    print(
        "\nzapisano dane dla "
        "kolejnego modulu:"
    )

    print(
        plik
    )

    print(
        "\nMODUL FILTROW "
        "DZIALA POPRAWNIE"
    )